# Text Similarity (STS) – Cosinus approach

## Project Overview

This notebook is part of a research project devoted to the **estimation of semantic textual similarity (STS)** using lexical-semantic resources and vector-based representations. The objective is to implement and compare several approaches for computing sentence-level semantic similarity by **combining standard lexical similarity measures with different aggregation strategies at the sentence level**. Three alternative aggregation strategies are explored in this project. Each approach is implemented in a different notebook:

1) **Notebook 1: Cosine similarity between semantic vectors derived from lexical similarity scores** as described in [Liu & Wang 2014](https://www.jsoftware.us/vol9/jsw0902-32.pdf) (this notebook)
2) **[Notebook 2](PLN-Proyecto%20-%20PARTE2%20-%20Ariadna%20con%20funciones%20wordnet.ipynb): A pairwise synset similarity aggregation strategy** (proposed in this project)
3) **[Notebook 3](PLN-Proyecto%20-%20PARTE3%20-%20Agirre%20con%20funciones%20wordnet.ipynb): Overlap similarity**  (based on the approach described in [Agirre et al. 2017](https://aclanthology.org/S17-2001/))


All notebooks in this project share the same experimental pipeline. The workflow includes **text preprocessing, synset assignment using WordNet (when applicable), computation of lexical similarity between tokens, and aggregation of these similarities into sentence-level similarity scores**. The notebooks differ in the formulation of the **sentence similarity stage**, while the preceding components remain identical in order to ensure comparability across experiments.

The evaluation is performed using sentence pairs from the **STSBenchmark** dataset, which provides human similarity judgments for pairs of English sentences. System scores produced by the implemented methods are compared with the gold-standard annotations using **Pearson correlation**.

Intermediate results and similarity scores generated by the notebooks were exported and analyzed externally. A summary of the experimental results is available in the [project report](../Procedimiento%20y%20discusión.pdf). Final analysis can be consulted in the [experiments comparison results file](../comparativa.xlsx).


## Subproject Overview – Cosinus approach

This notebook implements a **cosine-based model for sentence similarity**. In this approach, each sentence is represented as a **semantic vector constructed from lexical similarity scores** computed between the words (or synsets) of the two sentences, as described in [Liu & Wang 2014](https://www.jsoftware.us/vol9/jsw0902-32.pdf).
 
A joint representation space is first defined using the **union of lexical items or synsets appearing in both sentences**. For each dimension of this space, the value of a sentence vector corresponds to the **maximum lexical similarity between the corresponding lexical item and the items present in the sentence**.

Once the vectors associated with the two sentences are obtained, **sentence similarity is computed as the cosine similarity between these vectors**. This formulation allows lexical similarity scores derived from WordNet-based measures to be aggregated into a global sentence-level similarity score.

## Implementation

### Experiment configuration, dependencies, and auxiliary functions

In the following cells we set the **global configuration of the experiment** and load the required libraries. The main experimental parameters controlling the execution of the notebook are defined here. These include whether **Lesk-based word sense disambiguation** is applied, the type of **bag of words / bag of synsets** considered in the calculations, the **sentence-level similarity method** to be used, and the **corpus partition** to be processed.

Additional Boolean variables control whether libraries and data should be imported during execution. A descriptive string (`criterios`) is also created to identify the configuration of each experimental run when results are exported to an Excel file.

In [ ]:
# Parámetros a definir

    # desambiguar: True/False
        # True --> Synset se elige utilizando el desambiguador de Lesk
        # False --> Synset es el más común
    # expandir_contracciones: True/False (NO USADO)
        # True --> Se expanden las contracciones (don't --> do not)
        # False --> No se expanden las contracciones (don't --> don't) 
    # tipo_bow: --> Categorías de synsets que cogeremos para hacer los cálculos
        # todos (1) --> Todos los synsets
        # nv (2) --> Solo nombres y verbos
    # metodo_frase: lw / Ariadna / Agirre
        # lw --> distancia coseno según Liu-Wang
        # Ariadna --> 
        # Agirre --> 
    # partición: numero entero
        # fragmento del corpus que se procesa. Si se selecciona 1, se procesan todas las filas, 
        # si se selecciona 2, se procesan la mitad de filas, etc.
    # print_info: True/False
        # True --> Se muestra por pantalla la información de los parámetros seleccionados
        # False --> No se muestra por pantalla la información de los parámetros seleccionados
    # importar_librerias: True/False
        # True --> Se importan las librerías necesarias para el proyecto
        # False --> No se importan las librerías necesarias para el proyecto
    # importar_datos: True/False
        # True --> Se importan los datos necesarios para el proyecto
        # False --> No se importan los datos necesarios para el proyecto
        

desambiguar = True

expandir_contracciones = False

tipo_bow = 'nv'

metodo_frase = 'lw'

particion = 1

print_info = False

importar_librerias = True

importar_datos = True

criterios = 'Lesk-' + str(desambiguar) + ', tipo_bow-' + str(tipo_bow) + ', metodo-' + str(metodo_frase) + ', particion-' + str(particion)

print()
print('Desambiguación desambiguar:', desambiguar)
print('Tipo de BoW:', tipo_bow)
print('Método de comparación a nivel de frase:', metodo_frase)
print('Particion de filas a importar:', particion)
print('Importar librerías:', importar_librerias)
print('Importar datos:', importar_datos)

print(criterios)

In [ ]:
# Importación de librerías

if importar_librerias:
    
    import time
    import datetime

    import csv

    import pandas as pd
    pd.options.display.max_rows  # Para mostrar todas las columnas
    pd.set_option('display.max_colwidth', None)  # Para incluir todo el contenido de una celda, sin truncar contenido.
    pd.set_option('display.max_columns', 500)  # Para incluir todas las columnas (no serán más de 500)
    pd.set_option('display.max_rows', 6000)  # Para incluir todas las filas (no serán más de 6000)
    
    import re

    import nltk

    def ensure_nltk_resource(resource_path, download_name):
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(download_name)

    # Recursos necesarios de NLTK
    ensure_nltk_resource("corpora/wordnet", "wordnet")
    ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")
    ensure_nltk_resource("corpora/wordnet_ic", "wordnet_ic")
    ensure_nltk_resource("corpora/genesis", "genesis")
    ensure_nltk_resource("corpora/stopwords", "stopwords")

    from nltk.corpus import wordnet as wn

    from nltk.corpus import wordnet_ic
    ic_brown = wordnet_ic.ic('ic-brown.dat')
    ic_semcor = wordnet_ic.ic('ic-semcor.dat')

    from nltk.corpus import genesis
    ic_genesis = wn.ic(genesis, False, 0.0)
    
    if desambiguar == True:
        from nltk.wsd import lesk

import numpy as np

### Dataset import, cleaning and reduced corpus creation

The **STSBenchmark training dataset** is loaded into a pandas dataframe. The file contains pairs of English sentences annotated with **human similarity scores**, which serve as the reference values for evaluation. Duplicated sentence pairs are removed to avoid counting identical examples more than once during the experiments.

A reduced dataframe (`c`) is then created by selecting rows from the original corpus at regular intervals. This subset is used during development to **test preprocessing and similarity algorithms more quickly**, allowing faster iteration before running the full experiment on the complete dataset.

In [ ]:
# Importación de los datos del fichero sts-train.csv a un dataframe

start_total = time.time()

if importar_datos:

    file = '../stsbenchmark/sts-train.csv'
    corpus = pd.read_csv(file, sep='\t', on_bad_lines='skip', quoting=csv.QUOTE_NONE, usecols=range(7), header=None)
    corpus.head()
    corpus.columns = ['genre', 'file', 'year', 'id', 'gold_score', 'sent1', 'sent2']


end_total = time.time()

print('TIEMPO TRANSCURRIDO:', end_total-start_total)

corpus.sample(5)

In [ ]:
# Eliminación de duplicados

if importar_datos:

    corpus.drop_duplicates(['sent1', 'sent2'], keep='first', inplace=True)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

In [ ]:
# Creación de un corpus c reducido con filas representativas del corpus original

    # Para realizar pre-procesamientos que nos permitan verificar la adecuación de los algoritmos
    # en un tiempo reducido.

particion = particion

c = corpus.iloc[::particion, :].copy()
num_filas = 'Número de filas incluídas en el dataframe: ' + str(c.shape[0])

print(num_filas)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.head(12)

### Text preprocessing

Each sentence in the dataset undergoes the following standard natural language processing steps:  
1) tokenization  
2) lowercase conversion  
3) part-of-speech tagging*  
4) stopword removal (to retain only content-bearing words).

The resulting tokens are then prepared for the subsequent stages of the pipeline. The preprocessing results are stored in additional columns of the dataframe so that both the original sentences and their processed representations remain available.

A method for expanding contractions is implemented but not currently used (kept for potential future improvements).

In [ ]:
# Pre procesamiento

# ========================================
# TOKENIZACION, LIMPIEZA Y ANOTACIÓN
# ========================================

def expandir_contracciones_texto(texto):
    # specific
    texto = re.sub(r"won\'t", "will not", texto)
    texto = re.sub(r"can\'t", "can not", texto)

    # general
    texto = re.sub(r"n\'t", " not", texto)
    texto = re.sub(r"\'re", " are", texto)
    texto = re.sub(r"\'s", " is", texto)
    texto = re.sub(r"\'d", " would", texto)
    texto = re.sub(r"\'ll", " will", texto)
    texto = re.sub(r"\'t", " not", texto)
    texto = re.sub(r"\'ve", " have", texto)
    texto = re.sub(r"\'m", " am", texto)
    return texto

def tokenizar(sent, print_tokenizar =False):
    
    # tokens que aceptaremos (de nltk.org/book/ch03). Output: cadena de palabras.
   
    if print_tokenizar == True: print('Frase a tokenizar:', sent)
        
    pattern = r'''(?x)       # set flag to allow verbose regexps
        (?:[A-Z]\.)+         # abbreviations, e.g. U.S.A. ?: needs to be added to specify that the parenthesis specify the scope of the disjunction, not the selection o strings to be extracted
       | \w+(?:-\w+)*        # words with optional internal hyphens
       | [\$,\€]?\d+(?:\.\d+)?%?  # currency and percentages, e.g. $12.40, 82%
    '''
    if print_tokenizar == True: print('pattern:', pattern)
    '''
    AMPLIAR.
    - Analizar errores
    - Alguna solución para detectar phrasal verbs?
    '''

    # tokenización. Output: lista
   
    tokens = nltk.regexp_tokenize(sent, pattern, gaps=False)
    
    # limpieza 1: mayúsculas
    
    tokens = [token.lower() for token in tokens]   
    if print_tokenizar == True: print('Tokens:', tokens)
 
    # anotación POS. Output: lista de tuplas
    
    tagged_tokens = nltk.pos_tag(tokens, tagset='universal')
    
    # Convertimos las etiquetas gramaticales en notación Lesk;
        
    for i, tagged_token in enumerate(tagged_tokens):
        if tagged_tokens[i][1] == 'NOUN': tagged_tokens[i] = (tagged_tokens[i][0], 'n')
        if tagged_tokens[i][1] == 'VERB': tagged_tokens[i] = (tagged_tokens[i][0], 'v')
        if tagged_tokens[i][1] == 'ADJ': tagged_tokens[i] = (tagged_tokens[i][0], 'a')
        if tagged_tokens[i][1] == 'ADV': tagged_tokens[i] = (tagged_tokens[i][0], 'r')
    if print_tokenizar == True: print('Tagged_tokens:', tagged_tokens)
    
    # limipieza 2: stopwords - REVISAR si queremos/podemos incluir phrasal verbs, determinantes, pronombres...
            
                # Wordnet no incluye determinantes o preposiciones, de forma que "a" es asociado por Lesk 
                # al Synset('deoxyadenosine_monophosphate.n.01'), lo cual no nos conviene.
                # Por otro lado, Lesk utiliza para desambiguar solo las palabras que aparecen en Wordnet,
                # de modo que limpiar antes o después de Lesk las stopwords no debería afectar al resultado.
   
    if print_tokenizar == True: print('Limpieza de stopwords')
    stopwords = nltk.corpus.stopwords.words('english')
    tagged_stops = [tagged_stop for tagged_stop in tagged_tokens if tagged_stop[0] in stopwords]
    if print_tokenizar == True: print('Stopwords eliminadas:', tagged_stops)
    tokens = [token for token in tokens if token not in stopwords]
    tagged_tokens = [tagged_token for tagged_token in tagged_tokens if tagged_token[0] not in stopwords]
    if print_tokenizar == True: print('Tagged tokens sin stopwords:', tagged_tokens)

    return(tokens, tagged_tokens, tagged_stops)

# Ejemplo para comprobar funcionamiento tokenizar()
   
sent = "I'll always love my sweet baby"

print('****** EJEMPLO tokenizar() ********')
tokens = tokenizar(sent, print_tokenizar=True)
print('****')

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)


In [ ]:
c.head()

In [ ]:
# Tokenización de c

c['tokens1'], c['tokens2'] = None, None
c['tagged_tokens1'], c['tagged_tokens2'] = None, None
c['stops1'], c['stops2'] = None, None

print_tokenizar = False

for i in c.index:
    tok1 = tokenizar(c.at[i, 'sent1'], print_tokenizar)
    tok2 = tokenizar(c.at[i, 'sent2'], print_tokenizar)

    c.at[i, 'tokens1'] = tok1[0]
    c.at[i, 'tokens2'] = tok2[0]
    c.at[i, 'tagged_tokens1'] = tok1[1]
    c.at[i, 'tagged_tokens2'] = tok2[1]
    c.at[i, 'stops1'] = tok1[2]
    c.at[i, 'stops2'] = tok2[2]

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.head()

### Synset extraction

Preprocessed tokens are associated with **WordNet synsets** when possible. When enabled, **Lesk-based word sense disambiguation** is applied to select the most appropriate synset for each token; otherwise the first synset provided by WordNet is used. Tokens for which no synset is found are recorded separately.

In [ ]:
# ========================================
# 4. ASIGNACIÓN DE SYNSETS DE WORDNET
# ========================================

# Asignación de synsets 
    # Entrada: lista de tuplas con tokens y sus correspondientes etiquetas gramaticales
    # Argumentos: 
         # lesk: Opción de aplicar el algoritmo de Lesk  [Lesk 1986] de la librería wsd para desambiguar
                # Opción por defecto: n (no) --> Se asignará el primer synset del grupo de synsets (el más común)
                # Opción: s (sí) --> Asignar el synset determinado por Lesk
                
def syns(tagged_tokens, desambiguar=None):
    from nltk.wsd import lesk

    syns = list()
    errors = list()
    if desambiguar == True:
        for j in tagged_tokens:
            resultado_lesk = lesk([token for token, tag in tagged_tokens], j[0], j[1])
            if resultado_lesk is not None:
                syns.append(resultado_lesk)
            else:
                errors.append(j)
    else: 
        for j in tagged_tokens:
            try: syns.append(wn.synsets(j[0],j[1])[0])
            except: errors.append(j)
    return(syns, errors)
    
# Ejemplo para comprobar funcionamiento syns()

desambiguar_test = True

print('Desambiguación Lesk:', desambiguar_test )

tagged_tokens = c['tagged_tokens1'][151]
syns1 = syns(tagged_tokens, desambiguar_test )

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')

desambiguar_test = False

print('Desambiguación Lesk:', desambiguar_test )

tagged_tokens = c['tagged_tokens1'][151]
syns1 = syns(tagged_tokens, desambiguar_test )

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')

In [ ]:
# Asignacion de synsets a los tokens de c

desambiguar = desambiguar
print('Desambiguación Lesk:', desambiguar)
tipo_bow = tipo_bow
print('Tipo_bow:', tipo_bow)

c['syns1'] = None
c['syns2'] = None
c['errors1'] = None
c['errors2'] = None
c['syns1_nv'] = None
c['syns2_nv'] = None
c['syns1_resto'] = None
c['syns2_resto'] = None

for i in c.index:

    res1 = syns(c.at[i, 'tagged_tokens1'], desambiguar)
    res2 = syns(c.at[i, 'tagged_tokens2'], desambiguar)

    c.at[i,'syns1'] = res1[0]
    c.at[i,'syns2'] = res2[0]
    c.at[i,'errors1'] = res1[1]
    c.at[i,'errors2'] = res2[1]

    if tipo_bow == 'nv':
        c.at[i,'syns1_nv'] = [syn for syn in res1[0] if syn.pos() in ['n','v']]
        c.at[i,'syns2_nv'] = [syn for syn in res2[0] if syn.pos() in ['n','v']]
        c.at[i,'syns1_resto'] = [syn for syn in res1[0] if syn.pos() not in ['n','v']]
        c.at[i,'syns2_resto'] = [syn for syn in res2[0] if syn.pos() not in ['n','v']]

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.sample(5)

### Bag of words / bag of synsets construction

A **bag of words (BoW)** is constructed for each sentence pair by combining the lexical elements extracted from both sentences. In practice, this representation corresponds to a **bag of synsets**, which defines the **vector space used in the sentence similarity calculations**.

Depending on the experimental configuration, the representation may include **all synsets** or only those corresponding to **nouns and verbs**. Duplicate elements are removed to obtain a set of unique lexical items.

In [ ]:
# ========================================
# 5. BAG OF WORDS
# ========================================

# Creación de la Bag of Words (BoW) (de hecho, será una "Bag of synsets")
    # La necesitaremos luego como espacio vectorial para aplicar el método [Liu 2013] de similitud a nivel de frase.

tipo_bow = tipo_bow
print('Tipo_bow:', tipo_bow)
    
c['bow'] = c['syns1'] + c['syns2']
c['bow_errors'] =  c['errors1'] + c['errors2']

if tipo_bow == 'nv':
    c['bow_nv'] =  c['syns1_nv'] + c['syns2_nv']
    c['bow_resto'] =  c['syns1_resto'] + c['syns2_resto']

for i in c.index:
    c.at[i, 'bow'] = list(set(c.at[i, 'bow']))
    c.at[i, 'bow_errors'] = list(set(c.at[i, 'bow_errors']))    
    if tipo_bow == 'nv':
        c.at[i, 'bow_nv'] = list(set(c.at[i, 'bow_nv']))
        c.at[i, 'bow_resto'] = list(set(c.at[i, 'bow_resto']))

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.sample(5)

### Sentence vector construction

A method for sentence representation is defined using the **bag of synsets defined previously as the vector space**. For each dimension of this space, the value of the sentence vector corresponds to the **maximum lexical similarity between the associated synset and the synsets present in the sentence**. Lexical similarity between synsets is computed using different **WordNet-based similarity measures**. For IC-based measures (`res`, `jcn`, `lin`), the comparison is repeated using different information-content corpora.

Sentence vectors will later be computed according to this method when sentence similarity is evaluated.

In [ ]:
# ========================================
# 6. SIMILITUD A NIVEL LÉXICO - MÉTODOS INCLUÍDOS EN WORDNET
# ========================================

# Ejemplos de funciones de similitud integradas en Wordnet

    # ps --> path_similarity (POR DEFECTO). Rango: [0:1]
    # lch --> Leacock Chodorow. Rango: [0:3.64?] 
    # wup --> Wu_Palmer
    # res --> Resknik (requiere corpus ic)
    # jcn --> Jiang-Conrath (requiere corpus ic)
    # lin --> Lin (requiere corpus ic)

# Ejemplos para comprobar funcionamiento de los distintos métodos de simlitud de wordnet
    
syn1 = wn.synset('dog.n.01')
syn2 = wn.synset('cat.n.01')
syn3 = wn.synset('black.a.01')
syn4 = wn.synset('white.a.01')

try: print('wn.path_similarity(', syn1, syn1, '):',  wn.path_similarity(syn1, syn1))
except Exception as e: print('ERROR', e)
try: print('wn.path_similarity(', syn1, syn2, '):',  wn.path_similarity(syn1, syn2))
except Exception as e: print('ERROR', e)
try: print('wn.path_similarity(', syn1, syn3, '):',  wn.path_similarity(syn1, syn3))
except Exception as e: print('ERROR', e)
try: print('wn.path_similarity(', syn3, syn4, '):',  wn.path_similarity(syn3, syn4))
except Exception as e: print('ERROR', e)
    
print()

try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.lch_similarity(syn1, syn1))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.lch_similarity(syn1, syn2))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.lch_similarity(syn1, syn3))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.lch_similarity(syn3, syn4))
except Exception as e: print('ERROR', e)    

print()

try: print('wn.wup_similarity(', syn1, syn1, '):',  wn.wup_similarity(syn1, syn1))
except Exception as e: print('ERROR', e)
try: print('wn.wup_similarity(', syn1, syn2, '):',  wn.wup_similarity(syn1, syn2))
except Exception as e: print('ERROR', e)
try: print('wn.wup_similarity(', syn1, syn3, '):',  wn.wup_similarity(syn1, syn3))
except Exception as e: print('ERROR', e)
try: print('wn.wup_similarity(', syn3, syn4, '):',  wn.wup_similarity(syn3, syn4))
except Exception as e: print('ERROR', e)    

print()

ic = ic_brown
print('ic_brown')
try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.res_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.res_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.res_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.res_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.jcn_similarity(', syn1, syn1, '):',  wn.jcn_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn2, '):',  wn.jcn_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn3, '):',  wn.jcn_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn3, syn4, '):',  wn.jcn_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.lin_similarity(', syn1, syn1, '):',  wn.lin_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn2, '):',  wn.lin_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn3, '):',  wn.lin_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn3, syn4, '):',  wn.lin_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

ic = ic_semcor
print('ic_semcor')
try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.res_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.res_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.res_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.res_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.jcn_similarity(', syn1, syn1, '):',  wn.jcn_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn2, '):',  wn.jcn_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn3, '):',  wn.jcn_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn3, syn4, '):',  wn.jcn_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.lin_similarity(', syn1, syn1, '):',  wn.lin_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn2, '):',  wn.lin_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn3, '):',  wn.lin_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn3, syn4, '):',  wn.lin_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

ic = ic_genesis
print('ic_genesis')
try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.res_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.res_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.res_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.res_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.jcn_similarity(', syn1, syn1, '):',  wn.jcn_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn2, '):',  wn.jcn_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn3, '):',  wn.jcn_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn3, syn4, '):',  wn.jcn_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.lin_similarity(', syn1, syn1, '):',  wn.lin_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn2, '):',  wn.lin_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn3, '):',  wn.lin_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn3, syn4, '):',  wn.lin_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

### Sentence similarity computation - Cosine similarity

Sentence similarity is computed by constructing the vectors defined in the previous section and comparing them using **cosine similarity**. Each sentence is represented as a vector in the space defined by the **bag of synsets**, where vector values reflect the lexical similarity between sentence synsets and the corresponding dimensions.

The resulting cosine values constitute the **sentence-level similarity scores** produced by the Cosinus approach.

In [ ]:
# Funciones generales

# función coseno
    # Entrada: dos vectores
    # Salida: Un número real entre [0,1] (coseno entre los dos vectores)

def coseno(v1, v2):
    
    coseno = v1.dot(v2) / np.sqrt(v1.dot(v1) * v2.dot(v2))
    
    return(coseno)

# Ejemplo para comprobar funcionamiento coseno()

print('****** EJEMPLO coseno() ********')   
v1 = np.array([1,2,3])
v2 = np.array([1,0,1])
print('Coseno:', coseno(v1,v1))
print('Coseno:', coseno(v1,v2))
print('****')


In [ ]:
# ========================================
# SIMILITUD A NIVEL DE FRASE - LIU-WANG
# ========================================

# Vector semántico
    # Entrada: Un conjunto de synsets (frase) y una bag of words (bow) conteniendo el conjunto de synsets
        # si el parámetro "info" es "print", se mostrará información durante el proceso de cálculo
    # Argumentos: 
        # medida : función de similitud léxica que se quiere usar:
            # ps --> path_similarity (POR DEFECTO). 
                # Rango: [0:1]
            # lch --> Leacock Chodorow. 
                # Rango: [0:3.64?] 
            # wup --> Wu_Palmer
                # Rango: ]
            # res --> Resknik (requiere corpus ic)
                # Rango = [0-14,47]
                # Mismo synset --> Depende del synset (IC)
                # La palabra no está en el corpus --> infinito
                    # MODIFICACIÓN: Asignamos la media ponderada dentro del rango --> 9,04
            # jcn --> Jiang-Conrath (requiere corpus ic)
                # Rango = [0,1]
                # Mismo synset --> similitud = infinito (e+300)
                    # MODIFICACIÓN: Asignamos mismo synset --> similitud = 1
            # lin --> Lin (requiere corpus ic)
            # lw --> Liu-Wang
        # ic: corpus ic, para las funciones que lo requieran.
            # Intentar reproducir https://www.kdnuggets.com/2017/11/building-wikipedia-text-corpus-nlp.html
        # info: si 'print', se imprimirá información sobre el procesamiento.
    # Requerimientos: Haber importado wordnet_ic (si se utilizan las opciones res,jcn,lin)
    # Salida: Vector semántico del conjunto de synsets en el espacio definido por bow

def v_semantico(syns, bow, medida='ps', ic = '', info=''):
    v = np.zeros(len(bow), dtype=float)    

    if info == 'print': 
        print('bow:', bow)
        print('syns:', syns)
        print()
        print('Cálculo del vector semántico v correspondiente a syns en el espacio bow')
        print('v:', v)
        print()    
        print('Método de similitud léxica:', medida)
        print()       

    for index, synset in enumerate(bow):  
        if info == 'print': 
            print('Componente de v (index):', index)
            print('synset de bow correspondiente a index =',index, ':', synset)
            print('Valor inicial de la componente v', index, ':', v[index])
            print()
            print('Calculo del valor de la componente v', index)
            print()
        sims = list()
        for syn in syns:
            if info == 'print':
                print('Cálculo de la similitud de synset de bow con synset de syns:', synset, '<-->', syn)
                print()
            sim = 0.0
            if syn.pos() == synset.pos():
                if medida == 'ps':
                    sim = wn.path_similarity(syn, synset)
                elif medida == 'lch':
                    sim = wn.lch_similarity(syn, synset)
                elif medida == 'wup':
                    sim = wn.wup_similarity(syn, synset)
                elif medida == 'res':                    
                    sim = wn.res_similarity(syn, synset, ic)
                    if sim == 1.00000000e+300:
                        sim = 9.04
                elif medida == 'jcn':
                    if syn == synset:
                        sim = 1
                    else:
                        sim = wn.jcn_similarity(syn, synset, ic)
                elif medida == 'lin':
                    sim = wn.lin_similarity(syn, synset, ic)
                if info == 'print':
                    print('similitud:', sim)
                if sim is None: continue
                    
            else:
                sim = 0.0
                if info == 'print':
                    print(synset, 'y', syn, 'no pertenecen a la misma categoría gramatical')    
                    print('Similitud Liu ', synset, '<-->', syn, ':', sim)
            if not np.isnan(sim): 
                sims.append(sim)
            if info == 'print': print()
        v[index] = max(sims)
        if info == 'print':
            print('Valores de las similitudes:', sims)
            print("Valor de la componente v", index, '(máximo de las similitudes):', v[index])
            print()
    if info == 'print': print('v = ', v)
    if info == 'print': print()
    return(v)

    
# Ejemplo para comprobar funcionamiento v_semantico()

print('****** EJEMPLO v_semantico() ********')   

bow = c['bow'][501]
syns1 = c['syns1'][501]
syns2 = c['syns2'][501]
info = ''
ic = ''

print('bow:', bow)
print('syns1:', syns1)
print('syns2:', syns2)

print()
print('SIMILITUD LIU - WANG')

print()
print('SIMILITUDES DE WN QUE NO REQUIEREN IC')

medida = 'ps'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lch'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'wup'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

print()
print('SIMILITUDES DE WN QUE REQUIEREN IC. IC = BROWN')

ic = ic_brown

medida = 'res'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'jcn'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lin'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow,medida=medida, ic=ic, info=info))

print()
print('SIMILITUDES DE WN QUE REQUIEREN IC. IC = SEMCOR')
     
ic = ic_semcor

medida = 'res'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'jcn'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lin'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

print()
print('SIMILITUDES DE WN QUE REQUIEREN IC. IC = GENESIS')
     
ic = ic_genesis

medida = 'res'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'jcn'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lin'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))


print('****')

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

In [ ]:
# Similitud de frase Liu-Wang (cosino entre los vectores semánticos)
    # Entrada: Dos vectores semánticos
    # Salida: Similitud entre las dos frases, calculada como el coseno entre los dos vectores semánticos
    
def similitud_frase_lw(v1, v2):
        
    sim = coseno(v1, v2)
    
    return(sim)

# Ejemplo para comprobar funcionamiento similitud_frase_liu()

print('****** EJEMPLO similitud_frase() ********')   

v1 = v_semantico(syns1, bow, medida=medida, ic=ic, info=info)
v2 = v_semantico(syns2, bow, medida=medida, ic=ic, info=info)
print('Similitud a nivel de frase', similitud_frase_lw(v1, v2))
print('****')

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

In [ ]:
# Cálculo y almacenamiento de la puntuación Liu-Wang a nivel de frase

if metodo_frase == 'lw':

    c['v1_ps'] = None  
    c['v2_ps'] = None  

    c['v1_lch'] = None  
    c['v2_lch'] = None 

    c['v1_wup'] = None 
    c['v2_wup'] = None

    c['v1_res_brown'] = None 
    c['v2_res_brown'] = None 

    c['v1_res_semcor'] = None
    c['v2_res_semcor'] = None

    c['v1_res_genesis'] = None
    c['v2_res_genesis'] = None

    c['v1_jcn_brown'] = None 
    c['v2_jcn_brown'] = None 

    c['v1_jcn_semcor'] = None
    c['v2_jcn_semcor'] = None

    c['v1_jcn_genesis'] = None
    c['v2_jcn_genesis'] = None

    c['v1_lin_brown'] = None
    c['v2_lin_brown'] = None

    c['v1_lin_semcor'] = None
    c['v2_lin_semcor'] = None

    c['v1_lin_genesis'] = None
    c['v2_lin_genesis'] = None
    

    c['puntuacion_lw_ps'] = 0.0  
    c['puntuacion_lw_lch'] = 0.0 
    c['puntuacion_lw_wup'] = 0.0 
    c['puntuacion_lw_res_brown'] = 0.0
    c['puntuacion_lw_res_genesis'] = 0.0
    c['puntuacion_lw_res_semcor'] = 0.0
    c['puntuacion_lw_jcn_brown'] = 0.0 
    c['puntuacion_lw_jcn_semcor'] = 0.0
    c['puntuacion_lw_jcn_genesis'] = 0.0
    c['puntuacion_lw_lin_brown'] = 0.0
    c['puntuacion_lw_lin_semcor'] = 0.0
    c['puntuacion_lw_lin_genesis'] = 0.0

    for i in c.index:  
        gold_score = c.at[i, 'gold_score']
        sent1 = c.at[i, 'sent1']
        sent2 = c.at[i, 'sent2']
        syns1 = c.at[i, 'syns1']
        syns2 = c.at[i, 'syns2']
        if tipo_bow == 'nv':
            bow = c.at[i, 'bow_nv']
        else:
            bow = c.at[i, 'bow']
        start = time.time()
        if print_info == 's':
            print('FILA:', i)
            print('Gold score:', c.at[i, 'gold_score'])
            print('sent1:', c.at[i, 'sent1'])
            print('sent2:', c.at[i, 'sent2'])
            print('syns1:', c.at[i, 'syns1'])
            print('syns2:', c.at[i, 'syns2'])
            print('bow:', bow)

        try: c.at[i, 'v1_ps'] = v_semantico(c.at[i, 'syns1'], bow, medida='ps', ic='', info=info)
        except:  c.at[i, 'v1_ps'] = None
        try: c.at[i, 'v2_ps'] = v_semantico(c.at[i, 'syns2'], bow, medida='ps', ic='', info=info)
        except:  c.at[i, 'v2_ps'] = None
        try: c.at[i, 'puntuacion_lw_ps'] = similitud_frase_lw(c.at[i, 'v1_ps'], c.at[i, 'v2_ps'] )
        except:  c.at[i, 'puntuacion_lw_ps'] = None

        if print_info == 's':
            print('v1_ps', i, ': ', c.at[i, 'v1_ps'])
            print('v2_ps', i, ': ', c.at[i, 'v2_ps'])
            print('puntuacion_lw_ps', i, ': ', c.at[i, 'puntuacion_lw_ps'])

        try: c.at[i, 'v1_lch'] = v_semantico(c.at[i, 'syns1'], bow, medida='lch', ic='', info=info)
        except:  c.at[i, 'v1_lch'] = None
        try: c.at[i, 'v2_lch'] = v_semantico(c.at[i, 'syns2'], bow, medida='lch', ic='', info=info)
        except:  c.at[i, 'v2_lch'] = None
        try: c.at[i, 'puntuacion_lw_lch'] = similitud_frase_lw(c.at[i, 'v1_lch'], c.at[i, 'v2_lch'] )
        except:  c.at[i, 'puntuacion_lw_lch'] = None

        if print_info == 's':
            print('v1_lch', i, ': ', c.at[i, 'v1_lch'])
            print('v2_lch', i, ': ', c.at[i, 'v2_lch'])
            print('puntuacion_lw_lch', i, ': ', c.at[i, 'puntuacion_lw_lch'])

        try: c.at[i, 'v1_wup'] = v_semantico(c.at[i, 'syns1'], bow, medida='wup', ic='', info=info)
        except:  c.at[i, 'v1_wup'] = None
        try: c.at[i, 'v2_wup'] = v_semantico(c.at[i, 'syns2'], bow, medida='wup', ic='', info=info)
        except:  c.at[i, 'v2_wup'] = None
        try: c.at[i, 'puntuacion_lw_wup'] = similitud_frase_lw(c.at[i, 'v1_wup'], c.at[i, 'v2_wup'] )
        except:  c.at[i, 'puntuacion_lw_wup'] = None

        if print_info == 's':
            print('v1_wup', i, ': ', c.at[i, 'v1_wup'])
            print('v2_wup', i, ': ', c.at[i, 'v2_wup'])
            print('puntuacion_lw_wup', i, ': ', c.at[i, 'puntuacion_lw_wup'])

        ic = ic_brown

        try: c.at[i, 'v1_res_brown'] = v_semantico(c.at[i, 'syns1'], bow, medida='res', ic=ic_brown, info=info)
        except:  c.at[i, 'v1_res_brown'] = None
        try: c.at[i, 'v2_res_brown'] = v_semantico(c.at[i, 'syns2'], bow, medida='res', ic=ic_brown, info=info)
        except:  c.at[i, 'v2_res_brown'] = None
        try: c.at[i, 'puntuacion_lw_res_brown'] = similitud_frase_lw(c.at[i, 'v1_res_brown'], c.at[i, 'v2_res_brown'] )
        except:  c.at[i, 'puntuacion_lw_res_brown'] = None

        if print_info == 's':
            print('v1_res_brown', i, ': ', c.at[i, 'v1_res_brown'])
            print('v2_res_brown', i, ': ', c.at[i, 'v2_res_brown'])
            print('puntuacion_lw_res_brown', i, ': ', c.at[i, 'puntuacion_lw_res_brown'])      

        try: c.at[i, 'v1_jcn_brown'] = v_semantico(c.at[i, 'syns1'], bow, medida='jcn', ic=ic_brown, info=info)
        except:  c.at[i, 'v1_jcn_brown'] = None
        try: c.at[i, 'v2_jcn_brown'] = v_semantico(c.at[i, 'syns2'], bow, medida='jcn', ic=ic_brown, info=info)
        except:  c.at[i, 'v2_jcn_brown'] = None
        try: c.at[i, 'puntuacion_lw_jcn_brown'] = similitud_frase_lw(c.at[i, 'v1_jcn_brown'], c.at[i, 'v2_jcn_brown'] )
        except:  c.at[i, 'puntuacion_lw_jcn_brown'] = None

        if print_info == 's':
            print('v1_jcn_brown', i, ': ', c.at[i, 'v1_jcn_brown'])
            print('v2_jcn_brown', i, ': ', c.at[i, 'v2_jcn_brown'])
            print('puntuacion_lw_jcn_brown', i, ': ', c.at[i, 'puntuacion_lw_jcn_brown'])

        try: c.at[i, 'v1_lin_brown'] = v_semantico(c.at[i, 'syns1'], bow, medida='lin', ic=ic_brown, info=info)
        except:  c.at[i, 'v1_lin_brown'] = None
        try: c.at[i, 'v2_lin_brown'] = v_semantico(c.at[i, 'syns2'], bow, medida='lin', ic=ic_brown, info=info)
        except:  c.at[i, 'v2_lin_brown'] = None
        try: c.at[i, 'puntuacion_lw_lin_brown'] = similitud_frase_lw(c.at[i, 'v1_lin_brown'], c.at[i, 'v2_lin_brown'] )
        except:  c.at[i, 'puntuacion_lw_lin_brown'] = None

        if print_info == 's':
            print('v1_lin_brown', i, ': ', c.at[i, 'v1_lin_brown'])
            print('v2_lin_brown', i, ': ', c.at[i, 'v2_lin_brown'])
            print('puntuacion_lw_lin_brown', i, ': ', c.at[i, 'puntuacion_lw_lin_brown'])

        ic = ic_semcor

        try: c.at[i, 'v1_res_semcor'] = v_semantico(c.at[i, 'syns1'], bow, medida='res', ic=ic_semcor, info=info)
        except:  c.at[i, 'v1_res_semcor'] = None
        try: c.at[i, 'v2_res_semcor'] = v_semantico(c.at[i, 'syns2'], bow, medida='res', ic=ic_semcor, info=info)
        except:  c.at[i, 'v2_res_semcor'] = None
        try: c.at[i, 'puntuacion_lw_res_semcor'] = similitud_frase_lw(c.at[i, 'v1_res_semcor'], c.at[i, 'v2_res_semcor'])
        except:  c.at[i, 'puntuacion_lw_res_semcor'] = None

        if print_info == 's':
            print('v1_res_semcor', i, ': ', c.at[i, 'v1_res_semcor'])
            print('v2_res_semcor', i, ': ', c.at[i, 'v2_res_semcor'])
            print('puntuacion_lw_res_semcor', i, ': ', c.at[i, 'puntuacion_lw_res_semcor'])

        try: c.at[i, 'v1_jcn_semcor'] = v_semantico(c.at[i, 'syns1'], bow, medida='jcn', ic=ic_semcor, info=info)
        except:  c.at[i, 'v1_jcn_semcor'] = None
        try: c.at[i, 'v2_jcn_semcor'] = v_semantico(c.at[i, 'syns2'], bow, medida='jcn', ic=ic_semcor, info=info)
        except:  c.at[i, 'v2_jcn_semcor'] = None
        try: c.at[i, 'puntuacion_lw_jcn_semcor'] = similitud_frase_lw(c.at[i, 'v1_jcn_semcor'], c.at[i, 'v2_jcn_semcor'] )
        except:  c.at[i, 'puntuacion_lw_jcn_semcor'] = None

        if print_info == 's':
            print('v1_jcn_semcor', i, ': ', c.at[i, 'v1_jcn_semcor'])
            print('v2_jcn_semcor', i, ': ', c.at[i, 'v2_jcn_semcor'])
            print('puntuacion_lw_jcn_semcor', i, ': ', c.at[i, 'puntuacion_lw_jcn_semcor'])

        try: c.at[i, 'v1_lin_semcor'] = v_semantico(c.at[i, 'syns1'], bow, medida='lin', ic=ic_semcor, info=info)
        except:  c.at[i, 'v1_lin_semcor'] = None
        try: c.at[i, 'v2_lin_semcor'] = v_semantico(c.at[i, 'syns2'], bow, medida='lin', ic=ic_semcor, info=info)
        except:  c.at[i, 'v2_lin_semcor'] = None
        try: c.at[i, 'puntuacion_lw_lin_semcor'] = similitud_frase_lw(c.at[i, 'v1_lin_semcor'], c.at[i, 'v2_lin_semcor'] )
        except:  c.at[i, 'puntuacion_lw_lin_semcor'] = None

        if print_info == 's':
            print('v1_lin_semcor', i, ': ', c.at[i, 'v1_lin_semcor'])
            print('v2_lin_semcor', i, ': ', c.at[i, 'v2_lin_semcor'])
            print('puntuacion_lw_lin_semcor', i, ': ', c.at[i, 'puntuacion_lw_lin_semcor'])

        ic = ic_genesis

        try: c.at[i, 'v1_res_genesis'] = v_semantico(c.at[i, 'syns1'], bow, medida='res', ic=ic_genesis, info=info)
        except:  c.at[i, 'v1_res_genesis'] = None
        try: c.at[i, 'v2_res_genesis'] = v_semantico(c.at[i, 'syns2'], bow, medida='res', ic=ic_genesis, info=info)
        except:  c.at[i, 'v2_res_genesis'] = None
        try: c.at[i, 'puntuacion_lw_res_genesis'] = similitud_frase_lw(c.at[i, 'v1_res_genesis'], c.at[i, 'v2_res_genesis'] )
        except:  c.at[i, 'puntuacion_lw_res_genesis'] = None

        if print_info == 's':
            print('v1_res_genesis', i, ': ', c.at[i, 'v1_res_genesis'])
            print('v2_res_genesis', i, ': ', c.at[i, 'v2_res_genesis'])
            print('puntuacion_lw_res_genesis', i, ': ', c.at[i, 'puntuacion_lw_res_genesis'])

        try: c.at[i, 'v1_jcn_genesis'] = v_semantico(c.at[i, 'syns1'], bow, medida='jcn', ic=ic_genesis, info=info)
        except:  c.at[i, 'v1_jcn_genesis'] = None
        try: c.at[i, 'v2_jcn_genesis'] = v_semantico(c.at[i, 'syns2'], bow, medida='jcn', ic=ic_genesis, info=info)
        except:  c.at[i, 'v2_jcn_genesis'] = None
        try: c.at[i, 'puntuacion_lw_jcn_genesis'] = similitud_frase_lw(c.at[i, 'v1_jcn_genesis'], c.at[i, 'v2_jcn_genesis'] )
        except:  c.at[i, 'puntuacion_lw_jcn_genesis'] = None

        if print_info == 's':
            print('v1_jcn_genesis', i, ': ', c.at[i, 'v1_jcn_genesis'])
            print('v2_jcn_genesis', i, ': ', c.at[i, 'v2_jcn_genesis'])
            print('puntuacion_lw_jcn_genesis', i, ': ', c.at[i, 'puntuacion_lw_jcn_genesis'])

        try: c.at[i, 'v1_lin_genesis'] = v_semantico(c.at[i, 'syns1'], bow, medida='lin', ic=ic_genesis, info=info)
        except:  c.at[i, 'v1_lin_genesis'] = None
        try: c.at[i, 'v2_lin_genesis'] = v_semantico(c.at[i, 'syns2'], bow, medida='lin', ic=ic_genesis, info=info)
        except:  c.at[i, 'v2_lin_genesis'] = None
        try: c.at[i, 'puntuacion_lw_lin_genesis'] = similitud_frase_lw(c.at[i, 'v1_lin_genesis'], c.at[i, 'v2_lin_genesis'] )
        except:  c.at[i, 'puntuacion_lw_lin_genesis'] = None

        if print_info == 's':
            print('v1_lin_genesis', i, ': ', c.at[i, 'v1_lin_genesis'])
            print('v2_lin_genesis', i, ': ', c.at[i, 'v2_lin_genesis'])
            print('puntuacion_lw_lin_genesis', i, ': ', c.at[i, 'puntuacion_lw_lin_genesis'])    

        end = time.time()
        print(end-start)


    end_total = time.time()
    print('TIEMPO TRANSCURRIDO:', end_total-start_total)

### Results export

The resulting dataframe is exported to an Excel sheet for further analysis. Pearson correlation for each experiment is computed in [this Excel Worksheet]. Graphs are also provided for easy comparison.

In [ ]:
# Fix export problems with illegal characters in Excel 
import re

ILLEGAL_CHARACTERS_RE = re.compile(r'[\x00-\x08\x0B-\x0C\x0E-\x1F]')

def clean_illegal_chars(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub('', value)
    return value

c_export = c.apply(lambda col: col.map(clean_illegal_chars))

x = datetime.datetime.now()
c_export.to_excel(x.strftime("%y%m%d-%H%M-") + criterios + '.xlsx')